# ENN583 Week 10 — Modern Transformer Object Detectors

This companion practical introduces three transformer-based object detectors:

- **DETR** — the original end-to-end transformer detector.
- **RT-DETR** — a more recent detector designed for real-time inference.
- **Grounding DINO** — an open-vocabulary detector that finds objects described by text.

We will use the same images as the main object-detection practical. The models are loaded from Hugging Face and run one at a time so that this notebook can be used on a GPU with 4 GB of memory. CPU execution also works, but Grounding DINO will be slow.


## Learning objectives

By the end of this notebook, you should be able to:

1. Explain how object queries and set prediction differ from anchor-based detection.
2. Run DETR-family models and convert their normalized outputs into image coordinates.
3. Compare a standard closed-set detector with an open-vocabulary detector.
4. Investigate how confidence thresholds and text prompts affect detections.


## 1. Setup

The first execution downloads model weights and therefore requires internet access. Approximate downloads are 167 MB for DETR, 81 MB for RT-DETR R18, and 689 MB for Grounding DINO Tiny. Later runs use the Hugging Face cache.

The memory-conscious choices below are important on a 4 GB GPU: batch size 1, inference mode, mixed precision on CUDA, and only one model resident on the GPU at a time.


In [ ]:
# DETR uses timm for its convolutional backbone. Install it into the active
# notebook environment if it is not already available.
try:
    import timm
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'timm'])
    import timm

print(f'timm version: {timm.__version__}')


In [ ]:
# Standard-library tools for memory cleanup, file discovery, and timing.
import gc
import glob
import time
from pathlib import Path

# Plotting and image-loading utilities.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display
from PIL import Image
import torch

# Auto classes select the correct processor and model from a model ID.
from transformers import (
    AutoImageProcessor,
    AutoModelForObjectDetection,
    AutoModelForZeroShotObjectDetection,
    AutoProcessor,
)

# Prefer the GPU when one is available, but keep a CPU fallback.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')


In [ ]:
def clear_model_from_memory():
    # Release unused Python objects and cached CUDA memory between models.
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()


def draw_detections(image, boxes, scores, labels, title):
    # Draw model outputs in [xmin, ymin, xmax, ymax] pixel coordinates.
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(image)

    colours = plt.cm.tab10.colors
    for index, (box, score, label) in enumerate(zip(boxes, scores, labels)):
        # Detach from PyTorch and move to CPU before passing values to Matplotlib.
        xmin, ymin, xmax, ymax = box.detach().cpu().tolist()
        colour = colours[index % len(colours)]
        ax.add_patch(Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            fill=False, edgecolor=colour, linewidth=2
        ))
        ax.text(
            xmin, max(0, ymin - 4), f'{label}: {float(score):.2f}',
            color='white', fontsize=9,
            bbox={'facecolor': colour, 'alpha': 0.85, 'pad': 2}
        )

    ax.set_title(title)
    ax.axis('off')
    plt.show()


In [ ]:
# PIL supplies the RGB image format expected by Hugging Face processors.
image_path = Path('images/coco/000000017029.jpg')
image = Image.open(image_path).convert('RGB')
display(image)


## 2. DETR: detection as set prediction

Traditional detectors propose many candidate boxes and commonly use non-maximum suppression to remove duplicates. DETR instead predicts a fixed-size set using learned **object queries**. Bipartite matching is used during training so that each ground-truth object is assigned to one prediction.

The raw model output contains class logits and normalized boxes in `[centre_x, centre_y, width, height]` format. The image processor converts these into `[xmin, ymin, xmax, ymax]` pixel coordinates.


In [ ]:
# Download the processor configuration and pretrained COCO model.
detr_id = 'facebook/detr-resnet-50'
detr_processor = AutoImageProcessor.from_pretrained(detr_id)
detr = AutoModelForObjectDetection.from_pretrained(detr_id).to(device).eval()

# Resize, normalize, batch, and move the image tensor to the selected device.
detr_inputs = detr_processor(images=image, return_tensors='pt').to(device)
# Synchronization makes the CUDA timing and peak-memory reading meaningful.
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

start = time.perf_counter()
# Disable gradient tracking; use FP16 operations on CUDA to reduce memory use.
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
    detr_outputs = detr(**detr_inputs)
if device.type == 'cuda':
    torch.cuda.synchronize()
detr_time = time.perf_counter() - start

# Convert normalized centre-format boxes into pixel-space corner coordinates.
# The threshold removes low-confidence object queries.
detr_result = detr_processor.post_process_object_detection(
    detr_outputs, threshold=0.7,
    target_sizes=[(image.height, image.width)]
)[0]
# Translate numeric COCO class IDs into readable class names.
detr_labels = [detr.config.id2label[int(label)] for label in detr_result['labels']]

print('Raw class logits:', tuple(detr_outputs.logits.shape))
print('Raw normalized boxes:', tuple(detr_outputs.pred_boxes.shape))
print(f'Forward-pass time: {detr_time:.3f} s')
if device.type == 'cuda':
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3
    print(f'Peak allocated GPU memory: {peak_memory:.2f} GB')

draw_detections(
    image, detr_result['boxes'], detr_result['scores'], detr_labels,
    'DETR ResNet-50'
)


**Consider:** 
 * Why are there 100 raw predictions even when the image contains far fewer than 100 objects?
 * Inspect the raw outputs and compare them to the final displayed detections. What is the effect of the confidence threshold?  
 * What happens to the displayed detections as you reduce the confidence threshold?


In [ ]:
# Keep only one model on the GPU at a time.
# Remove references before asking PyTorch to release cached GPU memory.
del detr, detr_processor, detr_inputs, detr_outputs, detr_result
clear_model_from_memory()


## 3. RT-DETR: a practical modern detector

RT-DETR retains end-to-end transformer detection while redesigning the encoder and query selection for efficient inference. We use the small ResNet-18 checkpoint, which is particularly suitable for limited-memory teaching machines.

Run it on the same image so that the output, latency, and peak memory can be compared with DETR. Timing one forward pass is only illustrative: a proper speed benchmark requires warm-up runs and many repeated measurements.


In [ ]:
# R18 is the small ResNet-18 RT-DETR variant suited to limited GPU memory.
rtdetr_id = 'PekingU/rtdetr_r18vd'
rtdetr_processor = AutoImageProcessor.from_pretrained(rtdetr_id)
rtdetr = AutoModelForObjectDetection.from_pretrained(rtdetr_id).to(device).eval()

# Each model uses its own processor because its expected resize may differ.
rtdetr_inputs = rtdetr_processor(images=image, return_tensors='pt').to(device)
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

start = time.perf_counter()
# Perform the forward pass without storing gradients.
with torch.inference_mode(), torch.autocast(
    device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'
):
    rtdetr_outputs = rtdetr(**rtdetr_inputs)
if device.type == 'cuda':
    torch.cuda.synchronize()
rtdetr_time = time.perf_counter() - start

# Scale RT-DETR's normalized boxes back to the original image dimensions.
rtdetr_result = rtdetr_processor.post_process_object_detection(
    rtdetr_outputs, threshold=0.5,
    target_sizes=[(image.height, image.width)]
)[0]
rtdetr_labels = [
    rtdetr.config.id2label[int(label)] for label in rtdetr_result['labels']
]

print('Raw class logits:', tuple(rtdetr_outputs.logits.shape))
print('Raw normalized boxes:', tuple(rtdetr_outputs.pred_boxes.shape))
print(f'Forward-pass time: {rtdetr_time:.3f} s')
if device.type == 'cuda':
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3
    print(f'Peak allocated GPU memory: {peak_memory:.2f} GB')

draw_detections(
    image, rtdetr_result['boxes'], rtdetr_result['scores'], rtdetr_labels,
    'RT-DETR R18'
)


### Your turn: test a challenging image

Choose another image from `images/challenging`, run RT-DETR, and vary the confidence threshold. Look for missed detections, duplicate detections, and confident misclassifications.


In [ ]:
# Apply the already-loaded RT-DETR model to an out-of-distribution example.
challenging_path = Path('images/challenging/pikachu.jpeg')
challenging_image = Image.open(challenging_path).convert('RGB')

# Preprocessing must match the model used for the forward pass.
challenging_inputs = rtdetr_processor(
    images=challenging_image, return_tensors='pt'
).to(device)
with torch.inference_mode(), torch.autocast(
    device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'
):
    challenging_outputs = rtdetr(**challenging_inputs)

# Use a slightly lower threshold to inspect more uncertain predictions.
challenging_result = rtdetr_processor.post_process_object_detection(
    challenging_outputs, threshold=0.4,
    target_sizes=[(challenging_image.height, challenging_image.width)]
)[0]
challenging_labels = [
    rtdetr.config.id2label[int(label)]
    for label in challenging_result['labels']
]
draw_detections(
    challenging_image, challenging_result['boxes'],
    challenging_result['scores'], challenging_labels,
    f'RT-DETR — {challenging_path.name}'
)


In [ ]:
# Grounding DINO is larger, so remove RT-DETR before loading it.
del rtdetr, rtdetr_processor, rtdetr_inputs, rtdetr_outputs, rtdetr_result
del challenging_inputs, challenging_result, challenging_outputs
clear_model_from_memory()


## 4. Grounding DINO: open-vocabulary detection

The previous models use a fixed COCO class vocabulary. Grounding DINO instead receives an image and one or more text labels. This lets us ask for concepts that are not COCO categories. Separate thresholds control whether a box is retained and whether its alignment with the text is strong enough.

For multiple labels, pass a nested list such as `[['a dog', 'a person', 'a yellow toy']]`. Clear, concrete noun phrases usually work better than long sentences.


In [ ]:
# The processor handles both image preprocessing and text tokenization.
grounding_id = 'IDEA-Research/grounding-dino-tiny'
grounding_processor = AutoProcessor.from_pretrained(grounding_id)
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    grounding_id
).to(device).eval()


In [ ]:
# Unlike closed-set detectors, Grounding DINO receives the desired labels.
text_labels = ['a dog', 'a person', 'a sports ball', 'a frisbee']
# The outer list represents the batch; this batch contains one image.
grounded_inputs = grounding_processor(
    images=image, text=[text_labels], return_tensors='pt'
).to(device)
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

start = time.perf_counter()
# Run image and text through the multimodal detector together.
with torch.inference_mode(), torch.autocast(
    device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'
):
    grounded_outputs = grounding_model(**grounded_inputs)
if device.type == 'cuda':
    torch.cuda.synchronize()
grounded_time = time.perf_counter() - start

# Box and text thresholds control spatial confidence and text alignment.
grounded_result = grounding_processor.post_process_grounded_object_detection(
    grounded_outputs, grounded_inputs.input_ids,
    threshold=0.30, text_threshold=0.25,
    target_sizes=[(image.height, image.width)]
)[0]
# Transformers versions expose the matched phrases as text_labels or labels.
grounded_labels = grounded_result.get(
    'text_labels', grounded_result.get('labels', [])
)

print(f'Forward-pass time: {grounded_time:.3f} s')
if device.type == 'cuda':
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3
    print(f'Peak allocated GPU memory: {peak_memory:.2f} GB')

draw_detections(
    image, grounded_result['boxes'], grounded_result['scores'],
    grounded_labels, f'Grounding DINO — {text_labels}'
)


### Your turn: prompt an unusual object

The Pikachu image is useful because `Pikachu` is not one of the standard COCO detection categories. Try prompts such as `a yellow toy`, `a cartoon character`, `Pikachu`, and deliberately unrelated objects. Do different descriptions produce the same box? How do the score and false positives change?


In [ ]:
# Try several descriptions for the same visual concept.
prompt_image_path = Path('images/challenging/pikachu.jpeg')
prompt_image = Image.open(prompt_image_path).convert('RGB')
prompt_labels = ['Pikachu', 'a yellow toy', 'a cartoon character', 'a dog']

# Tokenize the candidate phrases alongside the image preprocessing.
prompt_inputs = grounding_processor(
    images=prompt_image, text=[prompt_labels], return_tensors='pt'
).to(device)
with torch.inference_mode(), torch.autocast(
    device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'
):
    prompt_outputs = grounding_model(**prompt_inputs)

# Lower thresholds expose more prompt-dependent successes and failures.
prompt_result = grounding_processor.post_process_grounded_object_detection(
    prompt_outputs, prompt_inputs.input_ids,
    threshold=0.25, text_threshold=0.20,
    target_sizes=[(prompt_image.height, prompt_image.width)]
)[0]
prompt_result_labels = prompt_result.get(
    'text_labels', prompt_result.get('labels', [])
)
draw_detections(
    prompt_image, prompt_result['boxes'], prompt_result['scores'],
    prompt_result_labels, f'Grounding DINO — {prompt_labels}'
)


### Optional exploration: the challenging-image collection

Running Grounding DINO over every image can be slow, especially on CPU. The following cell processes three examples. Increase the slice if time permits, or change the prompts to investigate a particular failure mode.


In [ ]:
# Limit the loop to three images so this remains practical on CPU.
challenging_paths = sorted(glob.glob('images/challenging/*'))[:3]
for file_name in challenging_paths:
    test_image = Image.open(file_name).convert('RGB')
    # Use the same prompt set for every image to make comparison easier.
    test_labels = ['a dog', 'a person']
    test_inputs = grounding_processor(
        images=test_image, text=[test_labels], return_tensors='pt'
    ).to(device)
    with torch.inference_mode(), torch.autocast(
        device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'
    ):
        test_outputs = grounding_model(**test_inputs)

    # Restore coordinates to the current image's original height and width.
    result = grounding_processor.post_process_grounded_object_detection(
        test_outputs, test_inputs.input_ids,
        threshold=0.30, text_threshold=0.25,
        target_sizes=[(test_image.height, test_image.width)]
    )[0]
    labels = result.get('text_labels', result.get('labels', []))
    draw_detections(
        test_image, result['boxes'], result['scores'], labels,
        f'Grounding DINO — {Path(file_name).name}'
    )


In [ ]:
# Release the final model and its remaining GPU tensors.
del grounding_model, grounding_processor
del grounded_inputs, grounded_outputs, grounded_result
del prompt_inputs, prompt_outputs, prompt_result
clear_model_from_memory()


## 5. Comparison and discussion

| Detector | Vocabulary | Main idea | Practical observation |
|---|---|---|---|
| Faster R-CNN / RetinaNet / FCOS | Fixed COCO classes | CNN-based two-stage, anchor-based, or point-based detection | Strong conventional baselines |
| DETR | Fixed COCO classes | Object queries and end-to-end set prediction | Conceptually important, but not designed primarily for speed |
| RT-DETR | Fixed COCO classes | Efficient end-to-end transformer detection | Best lightweight modern model in this practical |
| Grounding DINO | Text prompts | Align visual regions with language | Flexible vocabulary, but larger and prompt-sensitive |

Discuss the following:

1. Which detector found the dog most reliably in the challenging images?
2. Did the confidence scores mean the same thing across models?
3. When is a fixed vocabulary an advantage rather than a limitation?
4. What new failure modes appear when the detector is controlled by text?
5. Why is a single timed forward pass not a fair performance benchmark?


## If a 4 GB GPU runs out of memory

1. Restart the kernel and run only the section you need.
2. Confirm that the previous model-deletion cell has run.
3. Close other GPU applications and notebooks.
4. Reduce the input image dimensions before passing it to the processor.
5. Run Grounding DINO on CPU as a slower fallback.

Do not load all three models at once on a 4 GB GPU.
